# AirlineBookingsPreparation — Week 6: Airline Booking Data Preparation

**Instructions to students**

1. Do not delete or create cells.
2. Solve all questions and don't leave any questions empty.
3. Rename the Lab to `AirlineBookingsPreparation_StudentID`, changing `StudentID` to your actual student ID.
4. **Make sure to write correct variable names — it has marks.**
5. Where a question asks you to print, do it — printing is not marked, but it lets you check your own answers.
6. Keep `Airline_Bookings.csv` in the same folder as this notebook.
7. The questions build on each other, so run the cells in order.
8. For the data-preparation tasks, keep the requested variable names exactly as given.
9. Printing and displaying results are for your own checking unless a variable is explicitly named for grading.

**The story:** You are preparing airline booking data for a machine-learning workflow. You will load and clean the booking records, create useful encoded features, and prepare training and testing data with scaled numerical features.


## Part 0 — Setup (0 marks)

Import the libraries needed for this lab and print the versions of the main libraries used.

In [2]:
# TODO (0.1): Import pandas as pd, numpy as np, train_test_split, StandardScaler and MinMaxScaler
# TODO (0.2): Print the version of pandas, numpy and scikit-learn
# === Write your answer here: ===
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 3.1.0
numpy: 2.4.6
scikit-learn: 1.9.1


## Part 1 — Load, inspect and clean (3 marks)

### Q1 — Load, inspect and clean (3 marks)

Read `Airline_Bookings.csv` into `df`. Inspect the data by displaying the first 5 rows and by storing the missing-value count for each column in `missing_values` and the number of duplicate rows in `duplicate_count`, then print those diagnostics. **Remove duplicate rows from `df`**, then **fill missing `Booking_Lead_Days` and `Baggage_Kg` values with their respective column medians after duplicate removal**. Store the final shape in `cleaned_shape` and print it.

In [3]:
# TODO (1.1): Read Airline_Bookings.csv into a DataFrame -> df
# TODO (1.2): Display the first 5 rows of df
# TODO (1.3): Store the missing values in each column -> missing_values; store the duplicate-row count -> duplicate_count; print both
# TODO (1.4): Remove duplicate rows from df
# TODO (1.5): Fill missing Booking_Lead_Days and Baggage_Kg values with their column medians
# TODO (1.6): Store the shape of df after cleaning -> cleaned_shape; print it
# === Write your answer here: ===
df = pd.read_csv('Airline_Bookings.csv')
print(df.head(5))
print("------------------------------------------------------------------------------------")
missing_values = df.isnull().sum()
duplicate_count = df.duplicated().sum()
print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["Booking_Lead_Days"] = df["Booking_Lead_Days"].fillna(df["Booking_Lead_Days"].median())
df["Baggage_Kg"] = df["Baggage_Kg"].fillna(df["Baggage_Kg"].median())
cleaned_shape = df.shape
print("After cleaning:", cleaned_shape)

  Booking_ID  Distance_Km  Distance_Miles  Booking_Lead_Days  Baggage_Kg  \
0     BK0034         2870         1783.34                1.0        32.1   
1     BK0007         2233         1387.52               62.0        13.5   
2     BK0183         3343         2077.24               42.0         9.6   
3     BK0087         3421         2125.71               26.0         9.4   
4     BK0157         1582          983.01               32.0         6.3   

   Extra_Spend_OMR Cabin_Class Booking_Channel  Domestic_Flight  \
0            53.66     Economy             App                1   
1            10.57     Economy             App                1   
2            61.45    Business           Agent                1   
3           103.85     Economy             App                1   
4            27.98     Premium           Agent                1   

   Ticket_Price_OMR  
0            391.49  
1            288.68  
2            857.04  
3            435.60  
4            379.51  
--------

## Part 2 — Transform and encode (3 marks)

### Q2 — Transform and encode (3 marks)

Using the cleaned `df` from Q1, create the requested transformed features. Apply `np.log1p` to `Extra_Spend_OMR` and store it in a new `df` column named `Log_Extra_Spend`. Encode `Cabin_Class` as **Economy=0, Premium=1, Business=2** in a new column named `Cabin_Code`. Create one-hot encoded `Booking_Channel` columns with `pd.get_dummies(..., dtype=int)` and store them in `channel`, then add `channel` to `df` with `pd.concat`.

In [4]:
# TODO (2.1): Apply np.log1p to Extra_Spend_OMR and store the result in df -> Log_Extra_Spend
# TODO (2.2): Encode Cabin_Class as Economy=0, Premium=1, Business=2 -> Cabin_Code
# TODO (2.3): One-hot encode Booking_Channel using pd.get_dummies with dtype=int -> channel
# TODO (2.4): Add channel to df using pd.concat
# === Write your answer here: ===
df["Log_Extra_Spend"] = np.log1p(df["Extra_Spend_OMR"])
df["Cabin_Code"] = df["Cabin_Class"].map({"Economy": 0, "Premium": 1, "Business": 2})
channel = pd.get_dummies(df["Booking_Channel"], dtype=int)
df = pd.concat([df, channel], axis=1)



## Part 3 — Select, split and scale (3 marks)

### Q3 — Select, split and scale (3 marks)

Prepare the cleaned and encoded data for modelling. Drop `Booking_ID`, `Distance_Miles`, `Domestic_Flight`, `Extra_Spend_OMR`, `Cabin_Class` and `Booking_Channel` from `df` and store the result in `selected`. Then create `X` from all columns except `Ticket_Price_OMR` and `y` from `Ticket_Price_OMR`. Split them with **20% test data and `random_state=42`**, create a `StandardScaler` named `standard` and a `MinMaxScaler` named `minimum_maximum`, and make copies of the training and testing feature sets. Standardize `Distance_Km` into `Distance_Z` and normalize `Baggage_Kg` into `Baggage_MinMax`; fit each scaler on the training data and transform the testing data with the fitted scaler. Finally, drop the original `Distance_Km` and `Baggage_Kg` columns from the copied training and testing sets. Print the training/testing row counts and display the first 5 rows of both sets.

In [5]:
# TODO (3.1): Drop Booking_ID, Distance_Miles, Domestic_Flight, Extra_Spend_OMR, Cabin_Class and Booking_Channel -> selected
# TODO (3.2): Select all columns except Ticket_Price_OMR from selected -> X
# TODO (3.3): Select Ticket_Price_OMR from selected -> y
# TODO (3.4): Split X and y into X_train, X_test, y_train, y_test (20% test, random_state=42)
# TODO (3.5): Create a StandardScaler -> standard, and a MinMaxScaler -> minimum_maximum
# TODO (3.6): Make copies of X_train and X_test -> X_train_copy and X_test_copy
# TODO (3.7): Standardize Distance_Km: fit_transform on X_train_copy, transform on X_test_copy -> Distance_Z
# TODO (3.8): Normalize Baggage_Kg: fit_transform on X_train_copy, transform on X_test_copy -> Baggage_MinMax
# TODO (3.9): Drop Distance_Km and Baggage_Kg from X_train_copy and X_test_copy
# TODO (3.10): Print the training and testing row counts, and display the first 5 rows of both sets
# === Write your answer here: ===
# SELECT (leave out IDs, original categories, and original activity)
selected = df.drop(columns=["Booking_ID", "Distance_Miles", "Domestic_Flight", "Extra_Spend_OMR", "Cabin_Class", "Booking_Channel"])
X = selected.drop(columns=["Ticket_Price_OMR"])
y = selected["Ticket_Price_OMR"]

# SPLIT before learning scaling parameters
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TRANSFORM: fit on training data only
standard = StandardScaler()
minimum_maximum = MinMaxScaler()

X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

X_train_copy["Distance_Z"] = standard.fit_transform(X_train_copy[["Distance_Km"]]).ravel()
X_test_copy["Distance_Z"] = standard.transform(X_test_copy[["Distance_Km"]]).ravel()

X_train_copy["Baggage_MinMax"] = minimum_maximum.fit_transform(X_train_copy[["Baggage_Kg"]]).ravel()
X_test_copy["Baggage_MinMax"] = minimum_maximum.transform(X_test_copy[["Baggage_Kg"]]).ravel()

# Keep the transformed versions instead of the original columns
X_train_copy = X_train_copy.drop(columns=["Distance_Km", "Baggage_Kg"])
X_test_copy = X_test_copy.drop(columns=["Distance_Km", "Baggage_Kg"])

print("Training rows:", len(X_train_copy), "Testing rows:", len(X_test_copy))
display(X_train_copy.head())
display(X_test_copy.head())


Training rows: 192 Testing rows: 48


,Booking_Lead_Days,Log_Extra_Spend,Cabin_Code,Agent,App,Website,Distance_Z,Baggage_MinMax
139,76.0,3.203965,2,0,0,1,1.045083,0.021148
60,10.0,2.095561,0,0,1,0,-1.136991,0.706949
209,26.0,3.958334,0,0,0,1,-0.702622,0.250755
189,69.0,4.965080,0,1,0,0,0.505610,0.522659
212,49.0,3.719894,0,0,0,1,1.367836,0.356495


,Booking_Lead_Days,Log_Extra_Spend,Cabin_Code,Agent,App,Website,Distance_Z,Baggage_MinMax
24,79.0,3.340385,0,1,0,0,0.062871,0.604230
6,44.0,4.501364,0,1,0,0,-1.609494,0.495468
93,80.0,2.169054,1,0,1,0,-0.659837,0.465257
109,83.0,2.502255,1,0,1,0,1.441316,0.465257
104,8.0,4.337160,0,0,0,1,-0.314760,0.120846


---

You've reached the end of the lab. Please revise your answers before submitting, and make sure every variable name matches exactly what was asked.